# PRISM Precipitation Ingestion and Processing

This notebook prepares observed daily precipitation data from PRISM for extreme rainfall frequency analysis under climate non-stationarity.

Extreme rainfall events are defined consistently across time as **daily precipitation ≥ 3.0 inches (≈ 77 mm)**.  
(Threshold value can be adjusted by the user; see configuration below.)

---

## Purpose

- Ingest daily PRISM precipitation data
- Clean and standardize the time series
- Prepare outputs for downstream extreme-event analysis

This notebook focuses on **data preparation only**.  
Event flagging and frequency analysis occur in subsequent notebooks.

---

## Inputs

- PRISM daily precipitation CSV (user-provided)

---

## Outputs

- Cleaned daily precipitation time series saved to `outputs/tables/`

---

## Notes

- This notebook does **not** redistribute PRISM data.
- Users are responsible for complying with PRISM data use policies.
- All file paths are relative to the repository structure.


In [1]:
from pathlib import Path
import pandas as pd
import sys
import os

# Resolve repository root (notebooks/ is one level down)
REPO_ROOT = Path("..").resolve()

# Ensure src/ is importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Core repo directories
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATA_SAMPLE_DIR = DATA_DIR / "sample"

OUTPUTS_DIR = REPO_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
MAPS_DIR = OUTPUTS_DIR / "maps"

DOCS_DIR = REPO_ROOT / "docs"
SAMPLE_DIR = REPO_ROOT / "sample_data"

# Create expected output folders (safe if they already exist)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)


print("Repo root on path:", REPO_ROOT)
print("Notebook running successfully.")
print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("Outputs dir:", OUTPUTS_DIR)

Repo root on path: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework
Notebook running successfully.
Repo root: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework
Data dir: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\data
Outputs dir: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\outputs


In [2]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Paths to input data
PRISM_DATA_PATH = Path("../data/raw/prism_daily_precip.csv")
SAMPLE_PRISM_PATH = Path("../data/sample/prism_daily_demo.csv")

# Column names (must match input CSV)
DATE_COL = "Date"
PRECIP_COL = "inchesPpt"

# Extreme rainfall threshold (inches)
# Defined here for continuity; applied in later notebooks
EXTREME_THRESHOLD_IN = 3.0


In [3]:
# Select input source (real data preferred, sample as fallback)
if PRISM_DATA_PATH.exists():
    INPUT_PATH = PRISM_DATA_PATH
elif SAMPLE_PRISM_PATH.exists():
    INPUT_PATH = SAMPLE_PRISM_PATH
    print(f"Using sample data from {SAMPLE_PRISM_PATH}")
else:
    raise FileNotFoundError(
        "No PRISM input found.\n"
        "Provide either:\n"
        "- data/raw/prism_daily_precip.csv (real data)\n"
        "- data/sample/prism_daily_demo.csv (demo data)"
    )

Using sample data from ..\data\sample\prism_daily_demo.csv


In [4]:
from src.prism_extremes import load_prism_timeseries

df = load_prism_timeseries(
    INPUT_PATH,
    date_col=DATE_COL,
    precip_col=PRECIP_COL
)

df.head()

,inchesPpt
Date,
1981-01-01,0.00
1981-01-02,0.12
1981-06-15,2.45
1981-09-10,3.10
1982-03-04,0.00


In [5]:
print("Date range:", df.index.min(), "to", df.index.max())
print("Missing values:")
print(df.isna().sum())

Date range: 1981-01-01 00:00:00 to 2020-08-12 00:00:00
Missing values:
inchesPpt    0
dtype: int64


In [6]:
OUTPUT_PATH = Path("../outputs/tables/prism_daily_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH)

print(f"Cleaned PRISM data saved to {OUTPUT_PATH}")

Cleaned PRISM data saved to ..\outputs\tables\prism_daily_cleaned.csv


## Next Steps

This cleaned daily precipitation dataset is used in:

- `02_prism_extremes_analysis.ipynb`  
  → to flag extreme rainfall events and count annual frequency

Users may proceed directly to the next notebook after confirming this step completes successfully.
